# AML Observability Walkthrough

Dieses Notebook zeigt zwei Dinge: erstens den normalen Beyond-AI-Observability-PoC mit den eingecheckten Fixture-Daten und zweitens ein Missing-Data-Szenario, in dem eine sonst riskante Transaktion wegen fehlender Upstream-Daten nicht alertet.

In [ ]:
import json
from pathlib import Path

from observability.collector import InMemoryTraceCollector
from observability.models import CaseDisposition, TransactionRecord
from observability.pipeline import run_transaction
from observability.privacy import apply_trace_retention
from observability.queries import explain_what_changed, explain_why_flagged, explain_why_not_flagged
from observability.trace import TraceEventType

DATA_DIR = Path('../data').resolve()
transactions_path = DATA_DIR / 'synthetic_transactions.jsonl'
cases_path = DATA_DIR / 'synthetic_cases.jsonl'


In [ ]:
def read_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


transactions = [TransactionRecord.from_dict(row) for row in read_jsonl(transactions_path)]
cases = {
    row['transaction_id']: CaseDisposition.from_dict({k: v for k, v in row.items() if k != 'transaction_id'})
    for row in read_jsonl(cases_path)
}

len(transactions), sorted(cases)


In [ ]:
collector = InMemoryTraceCollector()
traces = []

for transaction in transactions:
    trace = run_transaction(
        transaction,
        collector,
        disposition=cases.get(transaction.transaction_id),
        auto_case_feedback=transaction.transaction_id not in cases,
    )
    traces.append(trace)

[(trace.trace_id, trace.has_alert(), len(trace.events)) for trace in traces]


In [ ]:
flagged_trace = next(trace for trace in traces if trace.has_alert())
non_alert_trace = next(trace for trace in traces if not trace.has_alert())

flagged_explanation = explain_why_flagged(flagged_trace)
non_alert_explanation = explain_why_not_flagged(non_alert_trace)

print(flagged_explanation.answer)
print(non_alert_explanation.answer)


In [ ]:
retention_decision, retained_trace = apply_trace_retention(non_alert_trace)
delta_explanation = explain_what_changed(traces[0], traces[1])

print(retention_decision)
print('retained event count:', len(retained_trace.events))
print(delta_explanation.answer)


## Missing-data / false-negative scenario

Wir erzeugen zwei sonst identische Transaktionen. Die erste enthaelt die risikorelevante Jurisdiktion, die zweite verliert genau dieses Feld upstream. Das Ziel ist nicht, ein perfektes Modell zu bauen, sondern zu zeigen, dass der Trace die Ursache des verpassten Alerts rekonstruierbar macht.

In [ ]:
complete_tx = TransactionRecord(
    transaction_id='walkthrough-complete',
    customer_id='cust-walkthrough',
    amount_eur=48000,
    customer_avg_monthly_eur=12000,
    beneficiary_jurisdiction='IRN',
    beneficiary_lei='529900T8BM49AURSDO55',
)

missing_jurisdiction_tx = TransactionRecord(
    transaction_id='walkthrough-missing-jurisdiction',
    customer_id='cust-walkthrough',
    amount_eur=48000,
    customer_avg_monthly_eur=12000,
    beneficiary_jurisdiction=None,
    beneficiary_lei='529900T8BM49AURSDO55',
)


In [ ]:
scenario_collector = InMemoryTraceCollector()
complete_trace = run_transaction(complete_tx, scenario_collector)
missing_trace = run_transaction(missing_jurisdiction_tx, scenario_collector)

complete_features = complete_trace.latest(event_type=TraceEventType.FEATURES_DERIVED).derived_features
missing_features = missing_trace.latest(event_type=TraceEventType.FEATURES_DERIVED).derived_features

complete_trace.has_alert(), missing_trace.has_alert()


In [ ]:
print(explain_why_flagged(complete_trace).answer)
print(explain_why_not_flagged(missing_trace).answer)
print(explain_what_changed(complete_trace, missing_trace).answer)

print('complete features:', complete_features)
print('missing-data features:', missing_features)
